# Dynamic RF IDK Cascades — Simple Streaming Research Notebook

This notebook implements the Random Forest skip-decision IDK cascade in small, visible steps.

It reads ImageNet-V2 from the local `ImageNet-V2 DataSet/` archives. Cached model outputs still save to `artifacts/` so expensive inference does not need to be repeated.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(exist_ok=True)

import time

import numpy as np

IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"

VARIANT_FOLDERS = {
    "matched-frequency": "imagenetv2-matched-frequency-format-val",
    "threshold-0.7": "imagenetv2-threshold0.7-format-val",
    "top-images": "imagenetv2-top-images-format-val",
}

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}


## 1. Stream ImageNet-V2 from local archives

Rows come from `ImageNet-V2 DataSet/*.tar.gz` with:

- PIL image
- archive path, including the class folder
- numeric ImageNet class label parsed from the folder name


In [ ]:
import tarfile
from PIL import Image

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples=None):
    archive = VARIANT_ARCHIVES[variant]
    emitted = 0

    with tarfile.open(archive, "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            if image_file is None:
                continue
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield {
                "image": image,
                "label": label_from_key(member.name),
                "key": member.name,
            }
            emitted += 1
            if max_samples is not None and emitted >= max_samples:
                break


def show_stream_examples(variant, n=3):
    for row in stream_imagenet_v2_rows(variant, max_samples=n):
        print(variant, row["label"], row["key"])


## 2. PyTorch iterable dataset

This wrapper applies the model transform and yields `(image_tensor, label, key)`.

In [38]:
def make_streaming_loader(variant, transform, batch_size=32, max_samples=None, num_workers=0):
    import torch
    from torch.utils.data import DataLoader, IterableDataset

    class ImageNetV2StreamDataset(IterableDataset):
        def __iter__(self):
            for row in stream_imagenet_v2_rows(variant, max_samples=max_samples):
                yield transform(row["image"]), row["label"], row["key"]

    return DataLoader(
        ImageNetV2StreamDataset(),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

## 3. Load pretrained classifiers

This notebook supports ResNet-18, ResNet-34, ResNet-50, ResNet-152, and ViT-B/16. The dynamic cascade uses ResNet-50 or ResNet-152 as its RF-selected heavy stage.


In [39]:
def load_model(model_name, device="cpu"):
    from torchvision import models

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        transform = weights.transforms()
    elif model_name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT
        model = models.resnet34(weights=weights)
        transform = weights.transforms()
    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        transform = weights.transforms()
    elif model_name == "resnet152":
        weights = models.ResNet152_Weights.DEFAULT
        model = models.resnet152(weights=weights)
        transform = weights.transforms()
    elif model_name == "vit_base_patch16_224":
        import timm
        from timm.data import create_transform, resolve_data_config

        model = timm.create_model(model_name, pretrained=True)
        transform = create_transform(**resolve_data_config({}, model=model))
    else:
        raise ValueError(f"Unknown model: {model_name}")

    model = model.to(device)
    model.eval()
    return model, transform


## 4. Random Forest skip model

Features from classifier A:

- confidence
- entropy
- top-1/top-2 margin

Labels:

- `0 = Skip B` when B would IDK
- `1 = Run B` when B would confidently predict

In [40]:
SKIP = 0
PREDICT = 1
CLASSIFICATION_THRESHOLD = 0.9
THRESHOLD_SKIP = 0.3


def load_cache(path):
    return dict(np.load(path, allow_pickle=False))


def confidence(cache):
    return cache["probabilities"].max(axis=1)


def skip_features(probabilities, eps=1e-12):
    conf = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + eps)).sum(axis=1)
    top_two = np.partition(probabilities, kth=-2, axis=1)[:, -2:]
    top_two.sort(axis=1)
    margin = top_two[:, 1] - top_two[:, 0]
    return np.column_stack([conf, entropy, margin]).astype(np.float32)


def build_skip_dataset(a_cache, b_cache, a_threshold=None, b_threshold=None):
    a_threshold = CLASSIFICATION_THRESHOLD if a_threshold is None else a_threshold
    b_threshold = CLASSIFICATION_THRESHOLD if b_threshold is None else b_threshold

    a_conf = confidence(a_cache)
    b_conf = confidence(b_cache)
    a_idk = a_conf < a_threshold

    X = skip_features(a_cache["probabilities"][a_idk])
    y = np.where(b_conf[a_idk] < b_threshold, SKIP, PREDICT).astype(np.int64)
    return X, y


def train_rf_skipper(training_pairs, a_threshold=None, b_threshold=None):
    from sklearn.ensemble import RandomForestClassifier

    X_parts = []
    y_parts = []
    for a_path, b_path in training_pairs:
        X, y = build_skip_dataset(load_cache(a_path), load_cache(b_path), a_threshold, b_threshold)
        X_parts.append(X)
        y_parts.append(y)

    X_train = np.concatenate(X_parts)
    y_train = np.concatenate(y_parts)

    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    rf.fit(X_train, y_train)
    print("training rows:", len(y_train), "skip:", int((y_train == SKIP).sum()), "predict:", int((y_train == PREDICT).sum()))
    return rf


## 5. Cascade evaluation

Compare no-skip, threshold-skip, and RF-skip policies from cached probabilities/timings.

In [41]:
def evaluate_cascade(a_cache, b_cache, c_cache, strategy, rf=None, a_threshold=None, b_threshold=None, threshold_skip=None):
    a_threshold = CLASSIFICATION_THRESHOLD if a_threshold is None else a_threshold
    b_threshold = CLASSIFICATION_THRESHOLD if b_threshold is None else b_threshold
    threshold_skip = THRESHOLD_SKIP if threshold_skip is None else threshold_skip

    labels = a_cache["labels"]
    final_preds = np.empty_like(labels)
    total_time = a_cache["times_ms"].astype(float).copy()

    a_conf = confidence(a_cache)
    b_conf = confidence(b_cache)

    a_successes = 0
    b_successes = 0
    skips = 0
    c_runs = 0
    skip_truth = []
    skip_pred = []

    for i in range(len(labels)):
        if a_conf[i] >= a_threshold:
            final_preds[i] = a_cache["predictions"][i]
            a_successes += 1
            continue

        b_would_idk = b_conf[i] < b_threshold
        should_skip = False

        if strategy == "threshold":
            should_skip = a_conf[i] < threshold_skip
        elif strategy == "rf":
            if rf is None:
                raise ValueError("RF strategy requires a trained rf model")
            features = skip_features(a_cache["probabilities"][i:i+1])
            should_skip = int(rf.predict(features)[0]) == SKIP
        elif strategy != "no-skip":
            raise ValueError("strategy must be no-skip, threshold, or rf")

        if strategy in {"threshold", "rf"}:
            skip_truth.append(SKIP if b_would_idk else PREDICT)
            skip_pred.append(SKIP if should_skip else PREDICT)

        if should_skip:
            skips += 1
            c_runs += 1
            total_time[i] += c_cache["times_ms"][i]
            final_preds[i] = c_cache["predictions"][i]
            continue

        total_time[i] += b_cache["times_ms"][i]
        if b_conf[i] >= b_threshold:
            b_successes += 1
            final_preds[i] = b_cache["predictions"][i]
        else:
            c_runs += 1
            total_time[i] += c_cache["times_ms"][i]
            final_preds[i] = c_cache["predictions"][i]

    result = {
        "strategy": strategy,
        "samples": len(labels),
        "accuracy": float((final_preds == labels).mean()),
        "time_ms_per_image": float(total_time.mean()),
        "a_successes": a_successes,
        "b_successes": b_successes,
        "skips": skips,
        "c_runs": c_runs,
    }

    if skip_truth:
        skip_truth = np.array(skip_truth)
        skip_pred = np.array(skip_pred)
        result["skip_decision_accuracy"] = float((skip_truth == skip_pred).mean())

    return result


## 6. Compare cached research runs

After you cache model outputs, train the skipper and evaluate policies.

In [42]:
from pathlib import Path

training_pairs = [
    (ARTIFACTS_DIR / "matched_resnet18.npz", ARTIFACTS_DIR / "matched_resnet34.npz"),
    (ARTIFACTS_DIR / "top_resnet18.npz", ARTIFACTS_DIR / "top_resnet34.npz"),
]

test_paths = {
    "a": ARTIFACTS_DIR / "threshold07_resnet18.npz",
    "b": ARTIFACTS_DIR / "threshold07_resnet34.npz",
    "c": ARTIFACTS_DIR / "threshold07_resnet152.npz",
}

required_paths = [path for pair in training_pairs for path in pair] + list(test_paths.values())
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    print("Missing cached model outputs:")
    for path in missing:
        print(" -", path)
    print("Run the matching cache_logits line before this comparison cell.")
    raise FileNotFoundError("Missing cached model outputs: " + ", ".join(missing))

# Train RF on Matched + Top. Report outcomes only on Threshold 0.7 test data.
# The C stage is the ResNet-152 cache.
rf = train_rf_skipper(training_pairs)

a_test = load_cache(test_paths["a"])
b_test = load_cache(test_paths["b"])
c_test = load_cache(test_paths["c"])

test_dataset_name = "threshold-0.7"
model_labels = ["A: resnet18", "B: resnet34", "C: resnet152"]
model_caches = [a_test, b_test, c_test]

cascade_results = [
    evaluate_cascade(a_test, b_test, c_test, strategy, rf=rf)
    for strategy in ["no-skip", "threshold", "rf"]
]

print("RF training data: matched-frequency + top-images")
print(f"Reported test data: {test_dataset_name}")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print(f"Static skip threshold: {THRESHOLD_SKIP}")
print("Final C model: resnet152")


training rows: 12597 skip: 10286 predict: 2311
RF training data: matched-frequency + top-images
Reported test data: threshold-0.7
Classification threshold: 0.9
Static skip threshold: 0.3
Final C model: resnet152


## 7. Printed threshold-0.7 outcomes

The output below is plain text and reports only the threshold-0.7 test data with the shared classification threshold.


In [43]:
print(f"Cascade outcomes on {test_dataset_name} test data")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print("Strategy    Samples  Accuracy  Time(ms/img)  A Accepted  B Accepted  Skips  C Runs  Skip Acc")
for result in cascade_results:
    skip_acc = result.get("skip_decision_accuracy")
    skip_acc_text = "n/a" if skip_acc is None else f"{skip_acc:.3f}"
    print(
        f"{result['strategy']:<11} "
        f"{result['samples']:>7}  "
        f"{result['accuracy']:.3f}     "
        f"{result['time_ms_per_image']:>10.2f}  "
        f"{result['a_successes']:>10}  "
        f"{result['b_successes']:>10}  "
        f"{result['skips']:>5}  "
        f"{result['c_runs']:>6}  "
        f"{skip_acc_text:>8}"
    )


Cascade outcomes on threshold-0.7 test data
Classification threshold: 0.9
Strategy    Samples  Accuracy  Time(ms/img)  A Accepted  B Accepted  Skips  C Runs  Skip Acc
no-skip       10000  0.789          18.33        3722        1179      0    5099       n/a
threshold     10000  0.789          17.80        3722        1130   1277    5148     0.376
rf            10000  0.789          17.25        3722         807   3880    5471     0.687


In [44]:
print(f"Routing counts on {test_dataset_name} test data")
print("Strategy    A Accepted  B Accepted  Direct Skips  C Runs")
for result in cascade_results:
    print(
        f"{result['strategy']:<11} "
        f"{result['a_successes']:>10}  "
        f"{result['b_successes']:>10}  "
        f"{result['skips']:>12}  "
        f"{result['c_runs']:>6}"
    )


Routing counts on threshold-0.7 test data
Strategy    A Accepted  B Accepted  Direct Skips  C Runs
no-skip           3722        1179             0    5099
threshold         3722        1130          1277    5148
rf                3722         807          3880    5471


In [45]:
import numpy as np

if "confidence" not in globals():
    def confidence(cache):
        return cache["probabilities"].max(axis=1)

if not all(name in globals() for name in ("a_test", "b_test", "c_test")):
    a_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet18.npz", allow_pickle=False))
    b_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet34.npz", allow_pickle=False))
    c_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet152.npz", allow_pickle=False))

test_dataset_name = "threshold-0.7"
model_labels = ["A: resnet18", "B: resnet34", "C: resnet152"]
model_caches = [a_test, b_test, c_test]

print(f"Standalone model outcomes on {test_dataset_name} test data")
print("Model          Samples  Accuracy  MeanConf  MeanTime(ms)")
for model_name, cache in zip(model_labels, model_caches):
    accuracy = float((cache["predictions"] == cache["labels"]).mean())
    mean_confidence = float(confidence(cache).mean())
    mean_time = float(cache["times_ms"].mean())
    print(
        f"{model_name:<14} "
        f"{len(cache['labels']):>7}  "
        f"{accuracy:.3f}     "
        f"{mean_confidence:.3f}     "
        f"{mean_time:>10.2f}"
    )


Standalone model outcomes on threshold-0.7 test data
Model          Samples  Accuracy  MeanConf  MeanTime(ms)
A: resnet18      10000  0.665     0.695           3.00
B: resnet34      10000  0.700     0.742           5.06
C: resnet152     10000  0.793     0.670          23.85


## 8. Current model skip-decision evaluation

Each skip-decision model predicts whether model B should be skipped on the threshold-0.7 test data with the shared classification threshold. Class 0 means Skip, and class 1 means Predict. The table below compares the Random Forest skipper against the static 0.3 threshold.


In [46]:
import numpy as np

SKIP = globals().get("SKIP", 0)
PREDICT = globals().get("PREDICT", 1)
CLASSIFICATION_THRESHOLD = globals().get("CLASSIFICATION_THRESHOLD", 0.9)
THRESHOLD_SKIP = globals().get("THRESHOLD_SKIP", 0.3)

if "confidence" not in globals():
    def confidence(cache):
        return cache["probabilities"].max(axis=1)

if "skip_features" not in globals():
    def skip_features(probabilities, eps=1e-12):
        conf = probabilities.max(axis=1)
        entropy = -(probabilities * np.log(probabilities + eps)).sum(axis=1)
        top_two = np.partition(probabilities, kth=-2, axis=1)[:, -2:]
        top_two.sort(axis=1)
        margin = top_two[:, 1] - top_two[:, 0]
        return np.column_stack([conf, entropy, margin]).astype(np.float32)

if not all(name in globals() for name in ("a_test", "b_test", "c_test")):
    a_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet18.npz", allow_pickle=False))
    b_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet34.npz", allow_pickle=False))
    c_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet152.npz", allow_pickle=False))

if "rf" not in globals():
    from sklearn.ensemble import RandomForestClassifier

    training_pairs = [
        (ARTIFACTS_DIR / "matched_resnet18.npz", ARTIFACTS_DIR / "matched_resnet34.npz"),
        (ARTIFACTS_DIR / "top_resnet18.npz", ARTIFACTS_DIR / "top_resnet34.npz"),
    ]
    training_features = []
    training_targets = []
    for a_path, b_path in training_pairs:
        a_train = dict(np.load(a_path, allow_pickle=False))
        b_train = dict(np.load(b_path, allow_pickle=False))
        a_idk_train = confidence(a_train) < CLASSIFICATION_THRESHOLD
        training_features.append(skip_features(a_train["probabilities"][a_idk_train]))
        training_targets.append(
            np.where(
                confidence(b_train)[a_idk_train] < CLASSIFICATION_THRESHOLD,
                SKIP,
                PREDICT,
            ).astype(np.int64)
        )

    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    rf.fit(np.concatenate(training_features), np.concatenate(training_targets))

test_dataset_name = "threshold-0.7"

def class_metrics(y_true, y_pred, class_id):
    tp = int(((y_true == class_id) & (y_pred == class_id)).sum())
    fp = int(((y_true != class_id) & (y_pred == class_id)).sum())
    fn = int(((y_true == class_id) & (y_pred != class_id)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def confusion_counts(y_true, y_pred):
    return np.array([
        [int(((y_true == SKIP) & (y_pred == SKIP)).sum()), int(((y_true == SKIP) & (y_pred == PREDICT)).sum())],
        [int(((y_true == PREDICT) & (y_pred == SKIP)).sum()), int(((y_true == PREDICT) & (y_pred == PREDICT)).sum())],
    ])


a_conf = confidence(a_test)
b_conf = confidence(b_test)
a_idk = a_conf < CLASSIFICATION_THRESHOLD

skip_y_true = np.where(b_conf[a_idk] < CLASSIFICATION_THRESHOLD, SKIP, PREDICT).astype(np.int64)
if len(skip_y_true):
    rf_y_pred = rf.predict(skip_features(a_test["probabilities"][a_idk])).astype(np.int64)
else:
    rf_y_pred = np.array([], dtype=np.int64)
threshold_y_pred = np.where(a_conf[a_idk] < THRESHOLD_SKIP, SKIP, PREDICT).astype(np.int64)

skip_eval_models = [
    ("Random Forest", rf_y_pred),
    (f"Threshold ({THRESHOLD_SKIP})", threshold_y_pred),
]

skip_metric_rows = []
skip_confusions = {}
for model_name, y_pred in skip_eval_models:
    accuracy = float((skip_y_true == y_pred).mean()) if len(skip_y_true) else 0.0
    skip_confusions[model_name] = confusion_counts(skip_y_true, y_pred)
    for class_id, class_name in [(SKIP, "Skip (0)"), (PREDICT, "Predict (1)")]:
        precision, recall, f1 = class_metrics(skip_y_true, y_pred, class_id)
        skip_metric_rows.append({
            "Model": model_name,
            "Class": class_name,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "Accuracy": accuracy,
        })

print(f"Skip-decision samples on {test_dataset_name}: {len(skip_y_true)}")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print("Model             Class        Precision  Recall  F1-Score  Accuracy")
for row_index, row in enumerate(skip_metric_rows):
    accuracy_text = f"{row['Accuracy']:.2f}" if row_index % 2 == 0 else ""
    print(
        f"{row['Model']:<17} {row['Class']:<12} "
        f"{row['Precision']:.2f}       {row['Recall']:.2f}    {row['F1-Score']:.2f}      {accuracy_text}"
    )

print()
print("Confusion matrices; rows=true class, columns=predicted class")
for model_name, matrix in skip_confusions.items():
    print(model_name)
    print("             Pred Skip  Pred Predict")
    print(f"True Skip    {matrix[0, 0]:>9}  {matrix[0, 1]:>12}")
    print(f"True Predict {matrix[1, 0]:>9}  {matrix[1, 1]:>12}")


Skip-decision samples on threshold-0.7: 6278
Classification threshold: 0.9
Model             Class        Precision  Recall  F1-Score  Accuracy
Random Forest     Skip (0)     0.90       0.69    0.78      0.69
Random Forest     Predict (1)  0.34       0.68    0.45      
Threshold (0.3)   Skip (0)     0.96       0.24    0.39      0.38
Threshold (0.3)   Predict (1)  0.23       0.96    0.37      

Confusion matrices; rows=true class, columns=predicted class
Random Forest
             Pred Skip  Pred Predict
True Skip         3508          1591
True Predict       372           807
Threshold (0.3)
             Pred Skip  Pred Predict
True Skip         1228          3871
True Predict        49          1130


In [ ]:
# Estimate the time to classify 5,000 images as one bulk RF-cascade workload via cache.
import time

import numpy as np

SKIP = globals().get("SKIP", 0)
PREDICT = globals().get("PREDICT", 1)
CLASSIFICATION_THRESHOLD = globals().get("CLASSIFICATION_THRESHOLD", 0.9)

if "confidence" not in globals():
    def confidence(cache):
        return cache["probabilities"].max(axis=1)

if "skip_features" not in globals():
    def skip_features(probabilities, eps=1e-12):
        conf = probabilities.max(axis=1)
        entropy = -(probabilities * np.log(probabilities + eps)).sum(axis=1)
        top_two = np.partition(probabilities, kth=-2, axis=1)[:, -2:]
        top_two.sort(axis=1)
        margin = top_two[:, 1] - top_two[:, 0]
        return np.column_stack([conf, entropy, margin]).astype(np.float32)

if not all(name in globals() for name in ("a_test", "b_test", "c_test")):
    a_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet18.npz", allow_pickle=False))
    b_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet34.npz", allow_pickle=False))
    c_test = dict(np.load(ARTIFACTS_DIR / "threshold07_resnet152.npz", allow_pickle=False))

if "rf" not in globals():
    from sklearn.ensemble import RandomForestClassifier

    training_pairs = [
        (ARTIFACTS_DIR / "matched_resnet18.npz", ARTIFACTS_DIR / "matched_resnet34.npz"),
        (ARTIFACTS_DIR / "top_resnet18.npz", ARTIFACTS_DIR / "top_resnet34.npz"),
    ]
    training_features = []
    training_targets = []
    for a_path, b_path in training_pairs:
        a_train = dict(np.load(a_path, allow_pickle=False))
        b_train = dict(np.load(b_path, allow_pickle=False))
        a_idk_train = confidence(a_train) < CLASSIFICATION_THRESHOLD
        training_features.append(skip_features(a_train["probabilities"][a_idk_train]))
        training_targets.append(
            np.where(
                confidence(b_train)[a_idk_train] < CLASSIFICATION_THRESHOLD,
                SKIP,
                PREDICT,
            ).astype(np.int64)
        )

    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    rf.fit(np.concatenate(training_features), np.concatenate(training_targets))

test_dataset_name = "threshold-0.7"
BULK_SAMPLE_COUNT = 5_000

available_samples = min(len(a_test["labels"]), len(b_test["labels"]), len(c_test["labels"]))
if available_samples < BULK_SAMPLE_COUNT:
    raise ValueError(
        f"Need {BULK_SAMPLE_COUNT} cached samples, but only {available_samples} are available."
    )

def first_n_samples(cache, sample_count):
    fields = ("probabilities", "labels", "predictions", "times_ms")
    return {field: cache[field][:sample_count] for field in fields}

bulk_a = first_n_samples(a_test, BULK_SAMPLE_COUNT)
bulk_b = first_n_samples(b_test, BULK_SAMPLE_COUNT)
bulk_c = first_n_samples(c_test, BULK_SAMPLE_COUNT)

routing_start = time.perf_counter()
a_confidence = confidence(bulk_a)
b_confidence = confidence(bulk_b)
a_accepted = a_confidence >= CLASSIFICATION_THRESHOLD
a_idk_indices = np.flatnonzero(~a_accepted)

direct_skips = np.zeros(BULK_SAMPLE_COUNT, dtype=bool)
if len(a_idk_indices):
    rf_decisions = rf.predict(skip_features(bulk_a["probabilities"][a_idk_indices]))
    direct_skips[a_idk_indices] = rf_decisions == SKIP

b_ran = ~a_accepted & ~direct_skips
b_accepted = b_ran & (b_confidence >= CLASSIFICATION_THRESHOLD)
c_ran = direct_skips | (b_ran & ~b_accepted)

final_predictions = np.empty(BULK_SAMPLE_COUNT, dtype=np.int64)
final_predictions[a_accepted] = bulk_a["predictions"][a_accepted]
final_predictions[b_accepted] = bulk_b["predictions"][b_accepted]
final_predictions[c_ran] = bulk_c["predictions"][c_ran]

total_time_ms = bulk_a["times_ms"].astype(float).copy()
total_time_ms[b_ran] += bulk_b["times_ms"][b_ran]
total_time_ms[c_ran] += bulk_c["times_ms"][c_ran]
routing_seconds = time.perf_counter() - routing_start

inference_seconds = total_time_ms.sum() / 1_000
estimated_total_seconds = inference_seconds + routing_seconds
throughput = BULK_SAMPLE_COUNT / estimated_total_seconds
bulk_accuracy = float((final_predictions == bulk_a["labels"]).mean())

print(f"Bulk RF-cascade classification on {test_dataset_name}")
print(f"Images: {BULK_SAMPLE_COUNT:,}")
print(f"Cached model inference time: {inference_seconds:.2f} seconds")
print(f"RF routing overhead: {routing_seconds:.2f} seconds")
print(f"Estimated total classification time: {estimated_total_seconds:.2f} seconds")
print(f"Estimated throughput: {throughput:.2f} images/second")
print(f"Accuracy: {bulk_accuracy:.3f}")
print(
    f"Routes: A accepted={int(a_accepted.sum())}, "
    f"B accepted={int(b_accepted.sum())}, "
    f"direct skips={int(direct_skips.sum())}, C runs={int(c_ran.sum())}"
)


Bulk RF-cascade classification on threshold-0.7
Images: 5,000
Cached model inference time: 87.68 seconds
RF routing overhead: 0.04 seconds
Estimated total classification time: 87.73 seconds
Estimated throughput: 57.00 images/second
Accuracy: 0.792
Routes: A accepted=1814, B accepted=402, direct skips=1954, C runs=2784


## Real-time cascade (running locally no cache for classification)


In [ ]:
# Real-time local cascade: actual model inference, no cached logits or timings.
from collections import Counter
import time

import numpy as np
import torch

REALTIME_SAMPLE_COUNT = 10000
REALTIME_VARIANT = "threshold-0.7"
REALTIME_DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

if "rf" not in globals():
    rf = train_rf_skipper([
        (ARTIFACTS_DIR / "matched_resnet18.npz", ARTIFACTS_DIR / "matched_resnet34.npz"),
        (ARTIFACTS_DIR / "top_resnet18.npz", ARTIFACTS_DIR / "top_resnet34.npz"),
    ])


def sync_device():
    if REALTIME_DEVICE.type == "mps":
        torch.mps.synchronize()
    elif REALTIME_DEVICE.type == "cuda":
        torch.cuda.synchronize()


def predict_local(model, transform, image):
    image = transform(image).unsqueeze(0).to(REALTIME_DEVICE)
    sync_device()
    start = time.perf_counter()
    with torch.inference_mode():
        probabilities = torch.softmax(model(image), dim=1)[0].detach().cpu().numpy()
    sync_device()
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    return probabilities, int(probabilities.argmax()), float(probabilities.max()), elapsed_ms


def run_realtime_local_cascade(rows, models_by_name, transforms_by_name, rf):
    labels = []
    final_predictions = []
    latencies_ms = []
    route_counts = Counter()
    model_times = Counter()
    model_runs = Counter()
    run_start = time.perf_counter()

    for row in rows:
        image = row["image"]
        label = int(row["label"])
        labels.append(label)
        sample_start = time.perf_counter()

        a_probs, a_pred, a_conf, a_ms = predict_local(models_by_name["resnet18"], transforms_by_name["resnet18"], image)
        model_times["resnet18"] += a_ms
        model_runs["resnet18"] += 1

        if a_conf >= CLASSIFICATION_THRESHOLD:
            route_counts["resnet18"] += 1
            final_prediction = a_pred
        elif int(rf.predict(skip_features(a_probs[None, :]))[0]) == SKIP:
            c_probs, c_pred, _, c_ms = predict_local(models_by_name["resnet152"], transforms_by_name["resnet152"], image)
            model_times["resnet152"] += c_ms
            model_runs["resnet152"] += 1
            route_counts["resnet152"] += 1
            final_prediction = c_pred
        else:
            b_probs, b_pred, b_conf, b_ms = predict_local(models_by_name["resnet34"], transforms_by_name["resnet34"], image)
            model_times["resnet34"] += b_ms
            model_runs["resnet34"] += 1
            if b_conf >= CLASSIFICATION_THRESHOLD:
                route_counts["resnet34"] += 1
                final_prediction = b_pred
            else:
                c_probs, c_pred, _, c_ms = predict_local(models_by_name["resnet152"], transforms_by_name["resnet152"], image)
                model_times["resnet152"] += c_ms
                model_runs["resnet152"] += 1
                route_counts["resnet152"] += 1
                final_prediction = c_pred

        final_predictions.append(final_prediction)
        latencies_ms.append((time.perf_counter() - sample_start) * 1000.0)

    total_seconds = time.perf_counter() - run_start
    labels = np.asarray(labels, dtype=np.int64)
    final_predictions = np.asarray(final_predictions, dtype=np.int64)
    latencies_ms = np.asarray(latencies_ms, dtype=np.float64)
    sample_count = len(labels)
    return {
        "samples": sample_count,
        "accuracy": float((final_predictions == labels).mean()),
        "wall_seconds": total_seconds,
        "throughput_fps": sample_count / total_seconds,
        "mean_latency_ms": float(latencies_ms.mean()),
        "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
        "route_counts": dict(route_counts),
        "model_runs": dict(model_runs),
        "total_model_time_ms": {name: float(model_times[name]) for name in models_by_name},
        "mean_model_time_ms": {name: model_times[name] / model_runs[name] for name in model_runs},
        "idle_time_ms_by_model": {
            name: max(0.0, total_seconds * 1000.0 - model_times[name])
            for name in models_by_name
        },
    }


model_a, transform_a = load_model("resnet18", REALTIME_DEVICE)
model_b, transform_b = load_model("resnet34", REALTIME_DEVICE)
model_c, transform_c = load_model("resnet152", REALTIME_DEVICE)
models_by_name = {"resnet18": model_a, "resnet34": model_b, "resnet152": model_c}
transforms_by_name = {"resnet18": transform_a, "resnet34": transform_b, "resnet152": transform_c}

realtime_rows = stream_imagenet_v2_rows(REALTIME_VARIANT, REALTIME_SAMPLE_COUNT)
realtime_result = run_realtime_local_cascade(realtime_rows, models_by_name, transforms_by_name, rf)

print("Real-time local RF cascade")
print(f"Dataset: {VARIANT_ARCHIVES[REALTIME_VARIANT]}")
print(f"Device: {REALTIME_DEVICE}")
print(f"Images: {realtime_result['samples']:,}")
print(f"Wall time: {realtime_result['wall_seconds']:.2f} seconds")
print(f"Throughput: {realtime_result['throughput_fps']:.2f} images/second")
print(f"Accuracy: {realtime_result['accuracy']:.3f}")
print(f"Mean latency: {realtime_result['mean_latency_ms']:.2f} ms")
print(f"P95 latency: {realtime_result['p95_latency_ms']:.2f} ms")
print("Routes:", realtime_result["route_counts"])
print("Model runs:", realtime_result["model_runs"])
print("Mean model time (ms):", realtime_result["mean_model_time_ms"])
print("Idle time by model (seconds):")
for model_name, idle_ms in realtime_result["idle_time_ms_by_model"].items():
    print(f"  {model_name}: {idle_ms / 1000.0:.3f}")
